In [3]:
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path
import sqlite3

In [2]:
county_month = pd.read_pickle("county.pkl")
county_month["STATE"] = county_month["STATE"].astype(str).str.strip()
county_month["COUNTY"] = county_month["COUNTY"].astype(str).str.strip().str.zfill(3)
county_month["year"] = county_month["year"].astype(int)
county_month["month"] = county_month["month"].astype(int)

NameError: name 'pd' is not defined

In [26]:
all_counties = (
    county_month[["STATE", "COUNTY"]]
    .drop_duplicates()
    .sort_values(["STATE", "COUNTY"])
    .reset_index(drop=True)
)

print(all_counties.shape)
all_counties.head()
#create the set of all counties

(5362, 2)


,STATE,COUNTY
0,AK,020
1,AK,110
2,AK,122
3,AK,130
4,AK,201


In [27]:
import pandas as pd

all_months = pd.date_range("1992-01-01", "2015-12-01", freq="MS")

month_df = pd.DataFrame({
    "date": all_months,
    "year": all_months.year,
    "month": all_months.month
})

print(month_df.shape)
month_df.head()
# generate all year-month

(288, 3)


,date,year,month
0,1992-01-01,1992,1
1,1992-02-01,1992,2
2,1992-03-01,1992,3
3,1992-04-01,1992,4
4,1992-05-01,1992,5


In [38]:
all_counties["key"] = 1
month_df["key"] = 1

full_panel = (
    all_counties.merge(month_df, on="key")
    .drop(columns="key")
    .sort_values(["STATE", "COUNTY", "year", "month"])
    .reset_index(drop=True)
)

print(full_panel.shape)
full_panel.head()
# full panel grid for each combo of state and year month 

(1544256, 5)


,STATE,COUNTY,date,year,month
0,AK,020,1992-01-01,1992,1
1,AK,020,1992-02-01,1992,2
2,AK,020,1992-03-01,1992,3
3,AK,020,1992-04-01,1992,4
4,AK,020,1992-05-01,1992,5


In [40]:
model_df = full_panel.merge(
    county_month,
    on=["STATE", "COUNTY", "year", "month"],
    how="left"
)

zero_fill_cols = [
    "fire_count",
    "notable_fire_count",
    "total_acres",
    "max_fire_size",
    "mean_fire_size",
    "median_fire_size",
]

for col in zero_fill_cols:
    model_df[col] = model_df[col].fillna(0)
model_df["target"] = (model_df["notable_fire_count"] > 0).astype(int)
# merge for full model 

In [37]:
county_month = (
    county_month
    .groupby(["STATE", "COUNTY", "year", "month"], as_index=False)
    .agg(
        fire_count=("fire_count", "sum"),
        notable_fire_count=("notable_fire_count", "sum"),
        total_acres=("total_acres", "sum"),
        max_fire_size=("max_fire_size", "max"),
        mean_fire_size=("mean_fire_size", "mean"),
        median_fire_size=("median_fire_size", "median"),
    )
)



In [47]:
model_df.to_pickle("model.pkl")

In [1]:

model_df.head()

NameError: name 'model_df' is not defined